# 063 — OCR y comprensión de documentos

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Pipeline OCR:** preprocesado (deskew, binarización) → análisis de layout (bloques,
tablas, orden de lectura) → segmentación en líneas → reconocimiento (CNN+RNN) →
postproceso (diccionarios, reglas). El reconocimiento moderno usa **CTC**: la red emite
un símbolo por columna de píxeles (incluido el blanco `∅`) y la pérdida colapsa
repeticiones y blancos para alinear con la etiqueta (`cc∅aas∅∅a → casa`), sin segmentar
caracteres.

**Métricas:** `CER = (S+D+I)/N` sobre caracteres (Levenshtein), WER sobre palabras. Un
CER global bajo puede ocultar errores concentrados en los campos críticos (importes,
identificadores).

**Comprensión de documentos:** extraer estructura (clave-valor, tablas, clase de
documento). LayoutLM añade a cada token su posición 2D en la página (y la imagen del
recorte), de modo que "el total" se encuentra combinando texto, geometría y estilo.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Alineación carácter a carácter: `O→0` (posición 2) y `5→S` (posición 9):
S = 2, D = 0, I = 0 → `CER = 2/12 ≈ 16.7 %`. Son confusiones de **forma glífica**: en
muchas fuentes `O`/`0` y `5`/`S` son casi idénticos; por eso el postproceso usa contexto
(en un importe se esperan dígitos, en una palabra letras).

**Ejercicio 2.** (a) `hh∅ooll∅a → h,∅,o,l,∅,a` colapsando repeticiones → `hola`.
(b) `h∅ol∅la → holla` (la doble `l` sobrevive porque hay un `∅` entre ambas).
(c) `ho∅∅laa → hola`. Sin el blanco separador, dos frames consecutivos con la misma letra
se colapsan a una sola: `∅` es el único mecanismo de CTC para emitir letras dobles.

**Ejercicio 3.** Leería a lo ancho: `La IA cambia` / `avanza rápido` — cada palabra es
correcta pero las frases quedan mezcladas: `"La IA cambia avanza rápido"`. Debía evitarlo
el **análisis de layout**, que detecta las dos columnas y fija el orden de lectura
(columna izquierda completa, luego derecha).

**Ejercicio 4.** Implementación con programación dinámica debajo; `levenshtein` devuelve
2 y `CER ≈ 0.167`.


In [ ]:
result = run_lab("perception", seed=63)
assert result["kind"] == "perception"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 4 — Levenshtein y CER verificados con código
def levenshtein(ref, hyp):
    m, n = len(ref), len(hyp)
    d = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        d[i][0] = i
    for j in range(n + 1):
        d[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            costo = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1,        # borrado
                          d[i][j - 1] + 1,        # inserción
                          d[i - 1][j - 1] + costo)  # sustitución
    return d[m][n]

def cer(ref, hyp):
    return levenshtein(ref, hyp) / len(ref)

ref, hyp = "TOTAL: 95,40", "T0TAL: 9S,40"
print("distancia:", levenshtein(ref, hyp))      # 2
print("CER:", round(cer(ref, hyp), 3))           # 0.167


In [ ]:
# Ejercicio 2 — colapso CTC verificado con código
def ctc_collapse(frames, blank="∅"):
    out, prev = [], None
    for f in frames:
        if f != prev and f != blank:
            out.append(f)
        prev = f
    return "".join(out)

print(ctc_collapse("h h ∅ o o l l ∅ a".split()))  # hola
print(ctc_collapse("h ∅ o l ∅ l a".split()))      # holla
print(ctc_collapse("h o ∅ ∅ l a a".split()))      # hola


## Reflexión

1. Un OCR reporta CER 1 % global sobre tus facturas. ¿Por qué eso no basta para automatizar
   el pago y qué métrica por campo definirías antes?
2. ¿Qué ventaja concreta aporta la posición 2D de cada token (LayoutLM) frente a pasar el
   texto plano del OCR a un modelo de lenguaje, en un documento con tabla de importes?
3. Si sustituyes el pipeline OCR por un VLM que "lee" la página, ¿qué nuevo modo de fallo
   introduces y cómo lo detectarías en producción?
